# Revocación

En este notebook se muestra como revocar tokens y leases de Vault

### Bootstrapping

In [31]:
%env WORKDIR=/tmp/vault        

env: WORKDIR=/tmp/vault


In [32]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next((directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()), None)
if ENV_FILE is None:
    raise FileNotFoundError("Could not find the persistent .env file")
load_dotenv(ENV_FILE)

VAULT_TOKEN = os.getenv('VAULT_TOKEN')
VAULT_ADDR = os.getenv('VAULT_ADDR')
VAULT_CACERT = os.getenv('VAULT_CACERT')

In [33]:
# Hashi only
!doormat login -f

import os
import subprocess

# Import the credentials produced by doormat into the notebook kernel.
result = subprocess.run(
    ["bash", "-lc", 'eval "$(doormat aws -a aws_jose.merchan_test export)" && env -0'],
    check=True,
    capture_output=True,
)
for entry in result.stdout.split(b"\0"):
    if entry.startswith(b"AWS_") and b"=" in entry:
        key, value = entry.split(b"=", 1)
        os.environ[key.decode()] = value.decode()

os.environ["AWS_REGION"] = "eu-central-1"

!aws eks update-kubeconfig --region eu-central-1 --name eks-infra-dev

INFO[0001] logging into doormat...                      
INFO[0002] successfully logged into doormat!            
Updated context arn:aws:eks:eu-central-1:492487827579:cluster/eks-infra-dev in /Users/jose/.kube/config


### Revocamos todas las leases asociadas a un role de Bases de Datos

In [34]:
! vault lease revoke -prefix database/creds/oracle-dynamic/

All revocation operations queued successfully!


### Verificamos que no existen leases

In [35]:
! vault list sys/leases/lookup/database/creds/oracle-dynamic/

No value found at sys/leases/lookup/database/creds/oracle-dynamic


### Creamos un nuevo lease y verificamos que podemos acceder a la base de datos con los credenciales asociados. Los credenciales se guardan en un fichero json

In [36]:
%%bash
set -euo pipefail

vault read -format=json database/creds/oracle-dynamic > /tmp/_creds.json
JSON=$(cat /tmp/_creds.json)
USER=$(jq -r '.data.username' <<<"$JSON")
PASSWORD=$(jq -r '.data.password' <<<"$JSON")
LEASE_ID=$(jq -r '.lease_id' <<<"$JSON")

echo "Dynamic role lease ID: $LEASE_ID"

LOGIN_RESULT=$(kubectl exec -i -n oracle oracle-db-0 -- /bin/bash -s -- "$USER" "$PASSWORD" <<'CONTAINER_SCRIPT'
set -euo pipefail
sqlplus -L -s /nolog <<SQL
WHENEVER SQLERROR EXIT SQL.SQLCODE
CONNECT $1/"$2"@//127.0.0.1:1521/FREEPDB1
SET HEADING OFF FEEDBACK OFF PAGES 0
SELECT 'DYNAMIC_LOGIN_OK' FROM dual;
EXIT
SQL
CONTAINER_SCRIPT
)


echo "Login del dynamic role: $(tr -d '[:space:]' <<<"$LOGIN_RESULT")"

Dynamic role lease ID: database/creds/oracle-dynamic/4FhmzCvQv9cbjchgWsD6Qchq
Login del dynamic role: DYNAMIC_LOGIN_OK


### Revocamos el lease individual

In [37]:
%%bash
JSON=$(cat /tmp/_creds.json)
LEASE_ID=$(jq -r '.lease_id' <<<"$JSON")
echo "Revoking lease ID: $LEASE_ID"
vault lease revoke $LEASE_ID

Revoking lease ID: database/creds/oracle-dynamic/4FhmzCvQv9cbjchgWsD6Qchq
All revocation operations queued successfully!


### Probamos a acceder con los credenciales previos sin crear un nuevo credencial

In [38]:
%%bash
set -euo pipefail


REVOKED_JSON=$(cat /tmp/_creds.json)
REVOKED_USER=$(jq -r '.data.username' <<<"$REVOKED_JSON")
REVOKED_PASSWORD=$(jq -r '.data.password' <<<"$REVOKED_JSON")
LEASE_ID=$(jq -r '.lease_id' <<<"$REVOKED_JSON")

echo "Dynamic role lease ID: $LEASE_ID"

set +e
LOGIN_RESULT=$(kubectl exec -i -n oracle oracle-db-0 -- /bin/bash -s -- "$REVOKED_USER" "$REVOKED_PASSWORD" <<'CONTAINER_SCRIPT'
set -euo pipefail
sqlplus -L -s /nolog <<SQL
WHENEVER SQLERROR EXIT SQL.SQLCODE
CONNECT $1/"$2"@//127.0.0.1:1521/FREEPDB1
SET HEADING OFF FEEDBACK OFF PAGES 0
SELECT 'REVOKED_LOGIN_UNEXPECTED' FROM dual;
EXIT
SQL
CONTAINER_SCRIPT
)
LOGIN_RC=$?
set -e


if [[ "$LOGIN_RC" -eq 0 ]]; then
  echo "ERROR: las credenciales revocadas todavía permiten login" >&2
  exit 1
fi
echo "OK: Oracle rechazó las credenciales después de revocar el lease."
unset REVOKED_JSON REVOKED_USER REVOKED_PASSWORD LOGIN_RESULT

Dynamic role lease ID: database/creds/oracle-dynamic/QpygouxdbYNxukMWMqzf41RW


command terminated with exit code 249


CalledProcessError: Command 'b'set -euo pipefail\n\n\nSTATIC_JSON=$(cat /tmp/static_creds.json)\nSTATIC_USER=$(jq -r \'.data.username\' <<<"$STATIC_JSON")\nSTATIC_PASSWORD=$(jq -r \'.data.password\' <<<"$STATIC_JSON")\nLEASE_ID=$(jq -r \'.lease_id\' <<<"$STATIC_JSON")\n\necho "Dynamic role lease ID: $LEASE_ID"\n\nLOGIN_RESULT=$(kubectl exec -i -n oracle oracle-db-0 -- /bin/bash -s -- "$STATIC_USER" "$STATIC_PASSWORD" <<\'CONTAINER_SCRIPT\'\nset -euo pipefail\nsqlplus -L -s /nolog <<SQL\nWHENEVER SQLERROR EXIT SQL.SQLCODE\nCONNECT $1/"$2"@//127.0.0.1:1521/FREEPDB1\nSET HEADING OFF FEEDBACK OFF PAGES 0\nSELECT \'STATIC_LOGIN_OK\' FROM dual;\nEXIT\nSQL\nCONTAINER_SCRIPT\n)\n\n\necho "Login del static role: $(tr -d \'[:space:]\' <<<"$LOGIN_RESULT")"\n'' returned non-zero exit status 249.

### Verificamos que no existen leases

In [ ]:
! vault list sys/leases/lookup/database/creds/oracle-dynamic/

No value found at sys/leases/lookup/database/creds/oracle-dynamic


# LDAP Auth Method

### Creamos una LDAP Database

In [39]:
%%bash
set -euo pipefail
kubectl apply -f manifest/openldap_deployment.yml
kubectl rollout status deployment/openldap -n vault --timeout=5m
kubectl wait --for=condition=Ready pod -l app.kubernetes.io/name=openldap \
  -n vault --timeout=2m

service/openldap created
deployment.apps/openldap created


In [40]:
! kubectl get pods -n vault

NAME                                    READY   STATUS    RESTARTS   AGE
openldap-5dbbc97c78-8lp8x               1/1     Running   0          47s
vault-0                                 1/1     Running   0          2d21h
vault-1                                 1/1     Running   0          2d21h
vault-2                                 1/1     Running   0          2d21h
vault-3                                 1/1     Running   0          2d21h
vault-4                                 1/1     Running   0          2d21h
vault-5                                 1/1     Running   0          2d21h
vault-agent-injector-65fcfc7599-6m9jq   1/1     Running   0          3d
vault-csi-provider-45b8p                2/2     Running   0          3d
vault-csi-provider-kpvjs                2/2     Running   0          3d
vault-csi-provider-q4wzl                2/2     Running   0          3d


## Creamos dos usuarios peter y alice asociados a distintos grupos

In [41]:
%%bash
export POD=$(kubectl get pods --selector=app.kubernetes.io/name=openldap -n vault -o json | jq -r '.items[0].metadata.name')

kubectl exec "$POD" -n vault -i -- \
  sh -c 'cat > /tmp/learn-vault-example.ldif' <<'EOF'
dn: ou=groups,dc=learn,dc=example
objectClass: organizationalunit
objectClass: top
ou: groups
description: groups of users

dn: ou=users,dc=learn,dc=example
objectClass: organizationalunit
objectClass: top
ou: users
description: users

dn: cn=dev,ou=groups,dc=learn,dc=example
objectClass: groupofnames
objectClass: top
description: testing group for dev
cn: dev
member: cn=alice,ou=users,dc=learn,dc=example

dn: cn=alice,ou=users,dc=learn,dc=example
objectClass: person
objectClass: top
sn: alice
memberOf: cn=dev,ou=groups,dc=learn,dc=example
userPassword: alice

dn: cn=ops,ou=groups,dc=learn,dc=example
objectClass: groupofnames
objectClass: top
description: testing group for ops
cn: ops
member: cn=peter,ou=users,dc=learn,dc=example

dn: cn=peter,ou=users,dc=learn,dc=example
objectClass: person
objectClass: top
sn: peter
memberOf: cn=ops,ou=groups,dc=learn,dc=example
userPassword: peter

dn: cn=serviceaccount,ou=users,dc=learn,dc=example
objectClass: person
objectClass: top
sn: serviceaccount
memberOf: cn=dev,ou=groups,dc=learn,dc=example
userPassword: serviceaccount
EOF

### Verificamos que el fichero existe

In [42]:
%%bash
export POD=$(kubectl get pods --selector=app.kubernetes.io/name=openldap -n vault -o json | jq -r '.items[0].metadata.name')
kubectl exec "$POD" -n vault -- \
  ls -l /tmp/learn-vault-example.ldif

-rw-r--r--. 1 root root 1081 Jul 27 11:24 /tmp/learn-vault-example.ldif


### Aplicamos la configuración

In [55]:
%%bash
set -euo pipefail
export POD=$(kubectl get pods --selector=app.kubernetes.io/name=openldap -n vault -o json | jq -r '.items[0].metadata.name')
set +e
kubectl exec "$POD" -n vault -- ldapadd -h 127.0.0.1 -p 389 \
  -w '2LearnVault' -c -x -D 'cn=admin,dc=learn,dc=example' \
  -f /tmp/learn-vault-example.ldif
LDAPADD_RC=$?
set -e
# ldapadd returns 68 when every requested DN already exists; that is idempotent success.
if [[ "$LDAPADD_RC" -ne 0 && "$LDAPADD_RC" -ne 68 ]]; then
  exit "$LDAPADD_RC"
fi
echo 'LDAP demo entries are present.'

ldap_add: Already exists (68)
ldap_add: Already exists (68)
ldap_add: Already exists (68)
ldap_add: Already exists (68)
ldap_add: Already exists (68)
ldap_add: Already exists (68)
ldap_add: Already exists (68)


adding new entry "ou=groups,dc=learn,dc=example"

adding new entry "ou=users,dc=learn,dc=example"

adding new entry "cn=dev,ou=groups,dc=learn,dc=example"

adding new entry "cn=alice,ou=users,dc=learn,dc=example"

adding new entry "cn=ops,ou=groups,dc=learn,dc=example"

adding new entry "cn=peter,ou=users,dc=learn,dc=example"

adding new entry "cn=serviceaccount,ou=users,dc=learn,dc=example"



command terminated with exit code 68


CalledProcessError: Command 'b'export POD=$(kubectl get pods --selector=app.kubernetes.io/name=openldap -n vault -o json | jq -r \'.items[0].metadata.name\')\nkubectl exec "$POD" -n vault -- \\\n  ldapadd \\\n    -h 127.0.0.1 \\\n    -p 389 \\\n    -w \'2LearnVault\' \\\n    -c -x \\\n    -D \'cn=admin,dc=learn,dc=example\' \\\n    -f /tmp/learn-vault-example.ldif\n'' returned non-zero exit status 68.

# Configuración en Vault

## Creamos namespace

In [89]:
%%bash
set -euo pipefail
if ! vault namespace lookup test >/dev/null 2>&1; then
  vault namespace create test
else
  echo 'Namespace test already exists'
fi

Key                Value
---                -----
custom_metadata    map[]
id                 6FbRP
path               test/


In [91]:
%%bash
set -euo pipefail
if ! VAULT_NAMESPACE=test vault auth list -format=json | jq -e 'has("ldap/")' >/dev/null; then
  VAULT_NAMESPACE=test vault auth enable ldap
else
  echo 'LDAP auth already enabled in namespace test'
fi

Success! Enabled ldap auth method at: ldap/


### Autorizamos el tráfico en el puerto LDAP entre nodos

In [62]:
%%bash
set -euo pipefail
SECURITY_GROUP_ID=sg-0950667285039d9f6
if ! aws ec2 describe-security-group-rules --region eu-central-1 \
  --filters "Name=group-id,Values=${SECURITY_GROUP_ID}" --output json | \
  jq -e --arg sg "$SECURITY_GROUP_ID" '.SecurityGroupRules[] | select(.IpProtocol == "tcp" and .FromPort == 389 and .ToPort == 389 and .ReferencedGroupInfo.GroupId == $sg)' >/dev/null; then
  aws ec2 authorize-security-group-ingress --region eu-central-1 \
    --group-id "$SECURITY_GROUP_ID" --protocol tcp --port 389 \
    --source-group "$SECURITY_GROUP_ID" >/dev/null
else
  echo 'LDAP ingress rule already exists'
fi

{
    "Return": true,
    "SecurityGroupRules": [
        {
            "SecurityGroupRuleId": "sgr-01a222cbc1c39d7f3",
            "GroupId": "sg-0950667285039d9f6",
            "GroupOwnerId": "492487827579",
            "IsEgress": false,
            "IpProtocol": "tcp",
            "FromPort": 389,
            "ToPort": 389,
            "ReferencedGroupInfo": {
                "GroupId": "sg-0950667285039d9f6",
                "UserId": "492487827579"
            },
            "SecurityGroupRuleArn": "arn:aws:ec2:eu-central-1:492487827579:security-group-rule/sgr-01a222cbc1c39d7f3"
        }
    ]
}


In [98]:
%%bash

VAULT_NAMESPACE=test vault write auth/ldap/config \
    binddn="cn=admin,dc=learn,dc=example" \
    bindpass="2LearnVault" \
    url="ldap://openldap.vault.svc.cluster.local:389" \
    userdn="ou=users,dc=learn,dc=example" \
    userattr="cn" \
    userfilter="({{.UserAttr}}={{.Username}})" \
    groupdn="dc=learn,dc=example"\
    groupfilter="(&(objectClass=person)(cn={{.Username}}))" \
    groupattr="memberOf"\
    insecure_tls=true

Success! Data written to: auth/ldap/config


## Creación de políticas

In [93]:
%%bash

cat >  $WORKDIR/dev_policy.hcl <<EOF
/*
path "secret/data/*" {
  capabilities = ["list"]
}
path "secret/metadata/*" {
  capabilities = [ "list"]
}
*/
path "secret/data/bu/org/cert" {
  capabilities = ["list"]
}

path "secret/data/bu/org/cert/*" {
  capabilities = ["list"]
}

path "secret/metadata/bu/org/cert" {
  capabilities = ["list", "read"]
}

path "secret/metadata/bu/org/cert/*" {
  capabilities = ["list", "read"]
}

path "secret/data/bu/org/cert/app/*" {
  capabilities = ["list", "read"]
}

path "secret/metadata/bu/org/cert/app/*" {
  capabilities = ["list", "read"]
}

/*
path "secret/data/dev" {
  capabilities = ["list"]
}
# Can create secret on secret/dev only
path "secret/data/dev/*" {
  capabilities = ["create", "update", "read", "list"]
}
path "secret/metadata/dev/*" {
  capabilities = ["read", "create", "update", "list"]
}
 
# List enabled secrets engine
path "sys/mounts" {
  capabilities = [ "read", "list" ]
}
*/

EOF

#---

cat >  $WORKDIR/ops_policy.hcl <<EOF

path "*" {
  capabilities = ["create", "read", "update", "delete", "list", "sudo"]
}

EOF

VAULT_NAMESPACE=test vault policy write dev $WORKDIR/dev_policy.hcl
VAULT_NAMESPACE=test vault policy write ops $WORKDIR/ops_policy.hcl


Success! Uploaded policy: dev
Success! Uploaded policy: ops


## Mapea Políticas a grupos de LDAP

In [94]:
%%bash
VAULT_NAMESPACE=test vault write auth/ldap/groups/dev policies=dev
VAULT_NAMESPACE=test vault write auth/ldap/groups/ops policies=ops

Success! Data written to: auth/ldap/groups/dev
Success! Data written to: auth/ldap/groups/ops


## Creamos Engine

In [95]:
%%bash
if ! VAULT_NAMESPACE=test vault secrets list -format=json | jq -e 'has("secret/")' >/dev/null; then
  VAULT_NAMESPACE=test vault secrets enable -path=secret kv-v2
else
  echo 'secret/ already enabled'
fi

Success! Enabled the kv-v2 secrets engine at: secret/


In [96]:
%%bash
VAULT_NAMESPACE=test vault kv put secret/bu/org/cert/app/secret1 value=secret1

=========== Secret Path ===========
secret/data/bu/org/cert/app/secret1

======= Metadata =======
Key                Value
---                -----
created_time       2026-07-27T11:57:55.93693458Z
custom_metadata    <nil>
deletion_time      n/a
destroyed          false
version            1


In [99]:
%%bash
VAULT_NAMESPACE=test vault login -method=ldap username=alice password=alice

WARNING! The VAULT_TOKEN environment variable is set! The value of this
variable will take precedence; if this is unwanted please unset VAULT_TOKEN or
update its value accordingly.



Success! You are now authenticated. The token information displayed below
is already stored in the token helper. You do NOT need to run "vault login"
again. Future Vault requests will automatically use this token.

Key                    Value
---                    -----
token                  hvs.CAESIEPKc2dKtoulMrz-I5fSVoSmP9tMmyROMMHpyw-X_8xMGigKImh2cy4wSGtkakFqejZ4REJWbTRpU3pzZ0xDMWEuNkZiUlAQoqUQ
token_accessor         eQZRJWuR5IISbzJ0DkYLljAM.6FbRP
token_duration         768h
token_renewable        true
token_policies         ["default" "dev"]
identity_policies      []
policies               ["default" "dev"]
token_meta_username    alice


### Leemos secretos para el que se tiene acceso

In [102]:
%%bash
VAULT_NAMESPACE=test vault login -method=ldap -format=json username=alice password=alice > $WORKDIR/alice.json
VAULT_TOKEN=$(jq -r '.auth.client_token' $WORKDIR/alice.json)

curl -s --header "X-Vault-Token: $VAULT_TOKEN"  --header "X-Vault-Namespace: test" \
  --request GET \
  $VAULT_ADDR/v1/secret/data/bu/org/cert/app/secret1 | jq -r 


WARNING! The VAULT_TOKEN environment variable is set! The value of this
variable will take precedence; if this is unwanted please unset VAULT_TOKEN or
update its value accordingly.



{
  "request_id": "1343244f-4378-0bdd-70c4-92ce20e09fb5",
  "lease_id": "",
  "renewable": false,
  "lease_duration": 0,
  "data": {
    "data": {
      "value": "secret1"
    },
    "metadata": {
      "created_time": "2026-07-27T11:57:55.93693458Z",
      "custom_metadata": null,
      "deletion_time": "",
      "destroyed": false,
      "version": 1
    }
  },
  "wrap_info": null,
  "warnings": null,
  "auth": null,
  "mount_type": "kv"
}


### Revocamos el token

In [103]:
%%bash

VAULT_NAMESPACE=test vault token revoke -mode=path auth/ldap

Success! Revoked token (if it existed)


## Verificamos que el token falla

In [104]:
%%bash
VAULT_TOKEN=$(jq -r '.auth.client_token' $WORKDIR/alice.json)

curl -s --header "X-Vault-Token: $VAULT_TOKEN"  --header "X-Vault-Namespace: test" \
  --request GET \
  $VAULT_ADDR/v1/secret/data/bu/org/cert/app/secret1 | jq -r 

{
  "errors": [
    "2 errors occurred:\n\t* permission denied\n\t* invalid token\n\n"
  ]
}


### Si el usuario se re-autentica obtendrá un nuevo token con lo que podrá acceder de nuevo

In [106]:
%%bash
VAULT_NAMESPACE=test vault login -method=ldap -format=json username=alice password=alice > $WORKDIR/alice.json
VAULT_TOKEN=$(jq -r '.auth.client_token' $WORKDIR/alice.json)

curl -s --header "X-Vault-Token: $VAULT_TOKEN"  --header "X-Vault-Namespace: test" \
  --request GET \
  $VAULT_ADDR/v1/secret/data/bu/org/cert/app/secret1 | jq -r 

WARNING! The VAULT_TOKEN environment variable is set! The value of this
variable will take precedence; if this is unwanted please unset VAULT_TOKEN or
update its value accordingly.



{
  "request_id": "5adf6c87-90d6-4fdc-6109-f4a9a23e15dd",
  "lease_id": "",
  "renewable": false,
  "lease_duration": 0,
  "data": {
    "data": {
      "value": "secret1"
    },
    "metadata": {
      "created_time": "2026-07-27T11:57:55.93693458Z",
      "custom_metadata": null,
      "deletion_time": "",
      "destroyed": false,
      "version": 1
    }
  },
  "wrap_info": null,
  "warnings": null,
  "auth": null,
  "mount_type": "kv"
}


# Namespace lock

In [108]:
! vault namespace lock test

Key           Value
---           -----
unlock_key    3ezUS6AD4lMq8WUfUHqmlmEC


### Tras bloquear el namespace ninguna interacción es posible

In [109]:
%%bash
VAULT_NAMESPACE=test vault login -method=ldap -format=json username=alice password=alice > $WORKDIR/alice.json
VAULT_TOKEN=$(jq -r '.auth.client_token' $WORKDIR/alice.json)

curl -s --header "X-Vault-Token: $VAULT_TOKEN"  --header "X-Vault-Namespace: test" \
  --request GET \
  $VAULT_ADDR/v1/secret/data/bu/org/cert/app/secret1 | jq -r 

Error authenticating: Error making API request.

Namespace: test/
URL: PUT https://vault.jose-merchan.sbx.hashidemos.io/v1/auth/ldap/login/alice
Code: 503. Errors:

* 1 error occurred:
	* API access to this namespace has been locked by an administrator - "test/" must be unlocked to gain access.




{
  "errors": [
    "permission denied"
  ]
}


### Namespace unlock

In [110]:
! vault namespace unlock test

### El usuario puede volver a acceder

In [111]:
%%bash
VAULT_NAMESPACE=test vault login -method=ldap -format=json username=alice password=alice > $WORKDIR/alice.json
VAULT_TOKEN=$(jq -r '.auth.client_token' $WORKDIR/alice.json)

curl -s --header "X-Vault-Token: $VAULT_TOKEN"  --header "X-Vault-Namespace: test" \
  --request GET \
  $VAULT_ADDR/v1/secret/data/bu/org/cert/app/secret1 | jq -r 

WARNING! The VAULT_TOKEN environment variable is set! The value of this
variable will take precedence; if this is unwanted please unset VAULT_TOKEN or
update its value accordingly.



{
  "request_id": "456b05db-f30a-88c0-5202-24ff32f45498",
  "lease_id": "",
  "renewable": false,
  "lease_duration": 0,
  "data": {
    "data": {
      "value": "secret1"
    },
    "metadata": {
      "created_time": "2026-07-27T11:57:55.93693458Z",
      "custom_metadata": null,
      "deletion_time": "",
      "destroyed": false,
      "version": 1
    }
  },
  "wrap_info": null,
  "warnings": null,
  "auth": null,
  "mount_type": "kv"
}


## RBAC

### En este escenario tenemos dos usuarios con políticas de acceso distintas sobre el namespace test

#### Peter por ejemplo puede crear engines

In [114]:
%%bash
VAULT_NAMESPACE=test vault login -method=ldap -format=json username=peter password=peter > $WORKDIR/peter.json
VAULT_TOKEN=$(jq -r '.auth.client_token' $WORKDIR/peter.json)

if ! VAULT_NAMESPACE=test vault secrets list -format=json | jq -e 'has("secret2/")' >/dev/null; then
  VAULT_TOKEN="$VAULT_TOKEN" vault secrets enable -path=secret2 -namespace=test kv-v2
else
  echo 'secret2/ already enabled'
fi

WARNING! The VAULT_TOKEN environment variable is set! The value of this
variable will take precedence; if this is unwanted please unset VAULT_TOKEN or
update its value accordingly.



Success! Enabled the kv-v2 secrets engine at: secret2/


#### Alice en cambio no

In [ ]:
%%bash
VAULT_NAMESPACE=test vault login -method=ldap -format=json username=alice password=alice > $WORKDIR/alice.json
VAULT_TOKEN=$(jq -r '.auth.client_token' $WORKDIR/alice.json)

set +e
OUTPUT=$(VAULT_TOKEN="$VAULT_TOKEN" vault secrets enable -path=secret3 -namespace=test kv-v2 2>&1)
RC=$?
set -e
if [[ "$RC" -eq 0 ]]; then
  echo 'ERROR: Alice pudo crear secret3/ sin autorización' >&2
  exit 1
fi
grep -q 'permission denied' <<<"$OUTPUT"
echo 'OK: RBAC denegó a Alice la creación de secret3/.'

WARNING! The VAULT_TOKEN environment variable is set! The value of this
variable will take precedence; if this is unwanted please unset VAULT_TOKEN or
update its value accordingly.

Error enabling: Error making API request.

Namespace: test/
URL: POST https://vault.jose-merchan.sbx.hashidemos.io/v1/sys/mounts/secret3
Code: 403. Errors:

* 1 error occurred:
	* permission denied




CalledProcessError: Command 'b"VAULT_NAMESPACE=test vault login -method=ldap -format=json username=alice password=alice > $WORKDIR/alice.json\nVAULT_TOKEN=$(jq -r '.auth.client_token' $WORKDIR/alice.json)\n\nvault secrets enable -path=secret3 -namespace=test kv-v2\n"' returned non-zero exit status 2.

# Clean Up

In [116]:
%%bash
set -u

WORKDIR="${WORKDIR:-/tmp/vault}"
SECURITY_GROUP_ID="sg-0950667285039d9f6"
cleanup_failed=0

echo "Cleaning up the Vault namespace..."
if vault namespace lookup test >/dev/null 2>&1; then
  # Unlock first in case execution stopped after the namespace-lock example.
  vault namespace unlock test >/dev/null 2>&1 || true
  VAULT_NAMESPACE=test vault token revoke -mode=path auth/ldap || cleanup_failed=1
  vault namespace delete test || cleanup_failed=1
else
  echo "Vault namespace test does not exist; skipping."
fi

echo "Removing the LDAP security-group ingress rule..."
RULE_IDS=$(aws ec2 describe-security-group-rules \
  --region eu-central-1 \
  --filters "Name=group-id,Values=${SECURITY_GROUP_ID}" \
  --output json | jq -r --arg sg "$SECURITY_GROUP_ID" '
    .SecurityGroupRules[]
    | select(
        .IpProtocol == "tcp"
        and .FromPort == 389
        and .ToPort == 389
        and .ReferencedGroupInfo.GroupId == $sg
      )
    | .SecurityGroupRuleId
  ')

if [[ -n "$RULE_IDS" ]]; then
  while IFS= read -r rule_id; do
    aws ec2 revoke-security-group-ingress \
      --region eu-central-1 \
      --group-id "$SECURITY_GROUP_ID" \
      --security-group-rule-ids "$rule_id" || cleanup_failed=1
  done <<< "$RULE_IDS"
else
  echo "LDAP ingress rule does not exist; skipping."
fi

echo "Removing OpenLDAP from Kubernetes..."
kubectl delete -f manifest/openldap_deployment.yml --ignore-not-found || cleanup_failed=1

echo "Removing temporary credential and policy files..."
rm -f \
  /tmp/_creds.json \
  /tmp/static_creds.json \
  "$WORKDIR/alice.json" \
  "$WORKDIR/peter.json" \
  "$WORKDIR/dev_policy.hcl" \
  "$WORKDIR/ops_policy.hcl"

if (( cleanup_failed != 0 )); then
  echo "Cleanup completed with errors; review the output above." >&2
  exit 1
fi

echo "Cleanup completed successfully."


Cleaning up the Vault namespace...
Success! Revoked token (if it existed)


WARNING! The following warnings were returned from Vault:

  * Namespace deletion has been queued. Progress will be reported in
  debug-level logs in Vault's server log.



Removing the LDAP security-group ingress rule...
{
    "Return": true,
    "RevokedSecurityGroupRules": [
        {
            "SecurityGroupRuleId": "sgr-01a222cbc1c39d7f3",
            "GroupId": "sg-0950667285039d9f6",
            "IsEgress": false,
            "IpProtocol": "tcp",
            "FromPort": 389,
            "ToPort": 389,
            "ReferencedGroupId": "sg-0950667285039d9f6"
        }
    ]
}
Removing OpenLDAP from Kubernetes...
service "openldap" deleted from vault namespace
deployment.apps "openldap" deleted from vault namespace
Removing temporary credential and policy files...
Cleanup completed successfully.
